# The Ultimate Beginner Guide to Tokenization (Using Your Verdict Book)

Hey — this notebook is for the **absolute beginner**.

No assumptions. No jargon overload. We’ll do this in tiny steps:
1. Understand what a token is with toy examples.
2. Build a tiny tokenizer ourselves (so you *really* get it).
3. Move from words to token IDs.
4. Learn why LLM tokenization is different.
5. Apply everything to your Verdict Book from `~/Downloads`.
6. Chunk and save outputs for future LLM workflows.


## How to use this notebook

- Read each markdown block.
- Run the code cell right below it.
- Inspect output before moving on.
- If anything feels confusing, re-run small examples and change inputs.

> Learning tokenization is like learning to chop vegetables before cooking a big meal.


## 0) Install dependencies (run once)

Uncomment if needed.


In [ ]:
# !pip install -q pypdf tiktoken pandas matplotlib


In [ ]:
from pathlib import Path
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt


## 1) Tokenization in one sentence

**Tokenization = splitting text into chunks a computer can process.**

Those chunks can be:
- words (`hello`, `world`)
- punctuation (`!`, `?`)
- subwords (`token`, `ization`)
- even characters (rare in modern LLM pipelines)


## 2) Super tiny example (manual splitting)

Let’s start with a baby sentence and split by spaces.


In [ ]:
toy_text = "I love pizza"
space_split = toy_text.split(" ")

print("Original:", toy_text)
print("Split by spaces:", space_split)
print("Number of tokens:", len(space_split))


### What just happened?

`split(" ")` means:
- find every space character
- cut there
- return pieces as a Python list

This is easy, but not perfect. Why?
- punctuation sticks to words (`hello!`)
- multiple spaces can be messy
- line breaks (`
`) are ignored weirdly


## 3) Slightly smarter tokenizer with regex

We’ll use `re.findall(...)` to keep only word-like pieces.


In [ ]:
example_text = "Hello, world! I paid $20 for 2 pizzas."

# \b\w+\b means: capture word boundaries around one-or-more word chars
word_tokens = re.findall(r"\b\w+\b", example_text.lower())

print("Original:", example_text)
print("Word tokens:", word_tokens)


### Function breakdown: `re.findall(pattern, text)`

- `re` = Python regex module
- `findall` = return all matches of a pattern
- `pattern` here: `r"\b\w+\b"`
  - `\w+` = one or more letters/numbers/underscore
  - `\b` = boundary between word and non-word
- `lower()` makes `Hello` and `hello` count as same token


## 4) Build a vocabulary (word -> ID)

Models don’t read strings directly; they read numbers.
So we map each unique token to an integer ID.


In [ ]:
def build_vocab(tokens):
    # Create sorted vocabulary dict: token -> integer ID
    unique_tokens = sorted(set(tokens))
    return {tok: i for i, tok in enumerate(unique_tokens)}


def encode_tokens(tokens, vocab):
    # Convert tokens to integer IDs using vocab dict
    return [vocab[tok] for tok in tokens]


def decode_ids(ids, reverse_vocab):
    # Convert integer IDs back to tokens
    return [reverse_vocab[i] for i in ids]


In [ ]:
toy_tokens = ["i", "love", "pizza", "pizza"]
vocab = build_vocab(toy_tokens)
reverse_vocab = {idx: tok for tok, idx in vocab.items()}
ids = encode_tokens(toy_tokens, vocab)
roundtrip = decode_ids(ids, reverse_vocab)

print("Tokens:", toy_tokens)
print("Vocab:", vocab)
print("Token IDs:", ids)
print("Decoded IDs:", roundtrip)


## 5) Why LLM tokenization is different

Word-level tokenization is great for learning.
But LLMs usually use **subword tokenization** to:
- handle rare words better
- support many languages
- keep vocabulary manageable

We’ll use `tiktoken` (common OpenAI-style tokenizer).


In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

sample = "tokenization is fun"
sample_ids = enc.encode(sample)
back_to_text = enc.decode(sample_ids)

print("Sample text:", sample)
print("LLM token IDs:", sample_ids)
print("Decoded text:", back_to_text)
print("LLM token count:", len(sample_ids))


## 6) Side-by-side comparison on tiny text


In [ ]:
tiny = "Tokenization helps LLMs understand text efficiently."

word_level = re.findall(r"\b\w+\b", tiny.lower())
llm_level = enc.encode(tiny)

print("Text:", tiny)
print("Word tokens:", len(word_level), word_level)
print("LLM tokens:", len(llm_level), llm_level)


## 7) Now apply to YOUR Verdict Book

### Step 7.1 Set file path
Edit `BOOK_FILENAME` if needed.


In [ ]:
DOWNLOADS_DIR = Path.home() / "Downloads"
BOOK_FILENAME = "Verdict Book.pdf"  # change if needed
book_path = DOWNLOADS_DIR / BOOK_FILENAME

print("Looking for:", book_path)
print("Exists?", book_path.exists())


### Optional helper: list likely files


In [ ]:
candidates = sorted([
    p.name for p in DOWNLOADS_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in {".pdf", ".txt", ".docx", ".epub"}
])

for name in candidates[:100]:
    print("-", name)


### Step 7.2 Load text from `.txt` or `.pdf`

Function walkthrough:
- check file extension
- if `.txt`: read directly
- if `.pdf`: extract page text with `pypdf`
- otherwise: raise clear error


In [ ]:
def load_book_text(path: Path) -> str:
    suffix = path.suffix.lower()

    if suffix == ".txt":
        return path.read_text(encoding="utf-8", errors="replace")

    if suffix == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(str(path))
        pages = []

        for i, page in enumerate(reader.pages, start=1):
            pages.append(page.extract_text() or "")
            if i % 50 == 0:
                print(f"Processed {i}/{len(reader.pages)} pages...")

        return "\n".join(pages)

    raise ValueError(f"Unsupported file type: {suffix}. Please use .txt or .pdf")


In [ ]:
raw_text = load_book_text(book_path)

print("Character count:", len(raw_text))
print("Preview:\n")
print(raw_text[:1200])


### Step 7.3 Clean text

Why clean?
- normalize curly quotes/dashes
- remove extra whitespace noise
- create more stable token counts


In [ ]:
def normalize_text(text: str) -> str:
    replacements = {
        "\u2018": "'", "\u2019": "'",
        "\u201c": '"', "\u201d": '"',
        "\u2013": "-", "\u2014": "-",
        "\u00a0": " ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"[ \t]+", " ", text)     # collapse repeated spaces/tabs
    text = re.sub(r"\n{3,}", "\n\n", text)   # collapse huge blank-line runs

    return text.strip()

clean_text = normalize_text(raw_text)
print("Cleaned character count:", len(clean_text))


## 8) Book tokenization — word level


In [ ]:
book_word_tokens = re.findall(r"\b\w+\b", clean_text.lower())
book_word_freq = Counter(book_word_tokens)

print("Word token count:", len(book_word_tokens))
print("Unique word tokens:", len(book_word_freq))
print("First 40 tokens:", book_word_tokens[:40])


In [ ]:
pd.DataFrame(book_word_freq.most_common(25), columns=["token", "count"])


In [ ]:
top_n = 20
df_top = pd.DataFrame(book_word_freq.most_common(top_n), columns=["token", "count"])

plt.figure(figsize=(12, 5))
plt.bar(df_top["token"], df_top["count"])
plt.title("Top Word Tokens in Verdict Book")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9) Book tokenization — LLM style (`tiktoken`)


In [ ]:
book_llm_ids = enc.encode(clean_text)

print("LLM token count:", len(book_llm_ids))
print("First 50 LLM token IDs:", book_llm_ids[:50])
print("Decoded snippet from first 80 IDs:\n")
print(enc.decode(book_llm_ids[:80]))


## 10) Compare counts (characters vs words vs LLM tokens)


In [ ]:
comparison = pd.DataFrame([
    {"metric": "characters", "value": len(clean_text)},
    {"metric": "word_tokens", "value": len(book_word_tokens)},
    {"metric": "unique_word_tokens", "value": len(book_word_freq)},
    {"metric": "llm_tokens_cl100k", "value": len(book_llm_ids)},
])
comparison


## 11) Chunking: split book into token-safe blocks

This matters because LLMs have context limits.


In [ ]:
def chunk_by_llm_tokens(text: str, encoding, max_tokens: int = 800, overlap: int = 100):
    # Split text into overlapping chunks by token count
    ids = encoding.encode(text)
    chunks = []
    start = 0

    while start < len(ids):
        end = start + max_tokens
        chunk_ids = ids[start:end]
        chunks.append(encoding.decode(chunk_ids))

        if end >= len(ids):
            break
        start = end - overlap

    return chunks

chunks = chunk_by_llm_tokens(clean_text, enc, max_tokens=800, overlap=100)
print("Number of chunks:", len(chunks))
print("First chunk preview:\n")
print(chunks[0][:1200])


## 12) Save outputs for reuse


In [ ]:
out_dir = Path("tokenization_outputs")
out_dir.mkdir(exist_ok=True)

(out_dir / "verdict_book_clean.txt").write_text(clean_text, encoding="utf-8")

pd.DataFrame(book_word_freq.items(), columns=["token", "count"]) \
  .sort_values("count", ascending=False) \
  .to_csv(out_dir / "word_frequencies.csv", index=False)

for i, chunk in enumerate(chunks, start=1):
    (out_dir / f"chunk_{i:04d}.txt").write_text(chunk, encoding="utf-8")

pd.DataFrame({"llm_token_id": book_llm_ids}).to_csv(out_dir / "all_llm_token_ids.csv", index=False)

print("Saved to:", out_dir.resolve())


## 13) Practice tasks (important)

1. Change `max_tokens` from 800 to 400 and observe chunk count.
2. Print a random chunk and estimate its topic.
3. Create a stopword filter and compare top-20 words before/after.
4. Try encoding names/entities from the book and inspect token IDs.
5. Compare token counts on raw vs cleaned text.


## 14) Recap (what you learned)

You now know how to:
- split text into word tokens
- map tokens to IDs manually
- use an LLM tokenizer (`tiktoken`)
- compare tokenization strategies
- chunk long text safely for LLM use
- export outputs for future NLP/LLM pipelines

If you want next, build: **Embeddings + Semantic Search over Verdict Book chunks**.
